## Things to try with CHARMM in the computer grafics course 

### Preliminary remarks

You should find the scripts referred to in this tutorial in the same directory (folder) where this notebook is located, i.e., under <tt>\<some path\>/charmm-stuff</tt>. **Please, here and in all CHARMM based tasks, also look at the comments in the scripts themselves.**

### 1. Structure of a basic CHARMM script

#### 1.1 Starting up CHARMM

While you can run CHARMM interactively, one usually tells the program what to do by means of a script. Under Unix this means that in order to execute a (short) CHARMM calculation, one runs from the command line (Unix Shell prompt, within JupyterHub the 'Terminal')

    charmm -i charmm_input_script.inp 

<font color="blue">(NB: this is an example, the <font color="red">file <tt>charmm_input_script.inp</tt> does not exist!</font>. Also, <font color="red">please make sure</font> that you know how to start a terminal in the JupyterHub (JH) environment and change (cd) into this directory!)</font>

Since as we shall see shortly CHARMM output tends to be verbose, one normally redirects the output to a file, thus ending up with

    charmm -i charmm_input_script.inp > charmm_output_file.out 

exploiting redirection (">") available under all Unix shells.

In the J(upyter)H(ub) environment you are working in, <tt>charmm_executable</tt> simply is <font color="blue"><tt>charmm</tt></font> and  <tt>charmm_input_script.inp</tt> and <tt>charmm_output_file.out</tt> have to be replaced by the names of the actual script/output file which you want to run (to which you want to save your output). 

In the following, I outline the most important elements of a CHARMM script


#### 1.2 Titles

First, let's do something really silly and start up charmm reading from an empty file; which can be easily accomplished by executing

> charmm -i /dev/null

in the Linux shell / command window / Terminal within the JupyterHub environment.

CHARMM prints a header telling you copyright info, version and some more stuff, followed by a warning

    RDTITL> No title read.
    
      ***** LEVEL  1 WARNING FROM <RDTITL> *****
      ***** Title expected.
      ******************************************
      BOMLEV (  0) IS NOT REACHED. WRNLEV IS  5

The job finishes by printing some status info. The interesting part is the warning from which we learn that CHARMM expected a "title". Indeed, each CHARMM script should start with a title, and if the main script tells CHARMM to read from another file, the program also expects to find a title at the beginning of that file. 

A title should not be confused with comments. E.g., it can only occur at the beginning of a file (we'll explain the apparent exceptions when we encounter them). Title lines start with a star/asterisk (*); to indicate the end of the title give a line containing only a star. (A title can consist of up to 32 consecutive lines) 
Thus,

    * This would be a short title
    * 

Make a text file containing just these two lines. Let's assume that you name the file <tt>title.inp</tt> and run it with CHARMM, i.e.,

> charmm -i title.inp

and among other stuff you should see the following in the output printed to the Linux console. 

    RDTITL> * THIS WOULD BE A SHORT TITLE
    RDTITL> *


HINT: <font color="blue">In many cases, you will start with an existing file; in this case just double click on the filename in the file manager of the left to start a simple editor. If you want to try out the above, you have to open a file from scratch. In the menu, click   
 **File->New->Text File**.
This opens a file 'untitled.txt', which most likely is not the name you want. Right-click on the 'tab' labeled 'untitled.txt', and you can change the file name (rename).</font>

#### 1.3 Comments 

Having blabbered so much about titles, what about comments: A comment in a CHARMM script is ''everything following an exclamation mark'' (!) I.e., 

   ! this is a comment on a line by itself

and this would be a line containing a CHARMM command, followed by a comment

   ENERgy ! as you might expect, this command calculates an energy

#### 1.4 Ending a CHARMM script

So far, CHARMM finished when it reached the end of the script file (as the line

                    NORMAL TERMINATION BY END OF FILE

in the output informs you). It's OK to end a CHARMM script in this manner, but the preferred way of stopping CHARMM is by issuing the 

    stop

command. We, thus, can create a first complete (though completely useless) CHARMM script, looking like 

    * A CHARMM script doing nothing
    *
      
    ! we really should be doing something, but in the meantime all we know is
      
    stop

In addition to the title, the comment is echoed as well. Note that CHARMM now prints upon finishing

                    NORMAL TERMINATION BY NORMAL STOP

**Concept check:** Put the stuff above in a file (e.g., <tt>myfile.inp</tt>), and have it execute by charmm. Make sure that 

    charmm -i myfile.inp

runs and ends as shown above, redirect the output

    charmm -i myfile.inp > myfile.out

and check that this worked / did what you expect!

### 2. A first full example

One of the major hurdles in getting started with CHARMM is the initial setting up of one's simulation system (usually starting from a PDB file of the system (protein) of interest). We'll dwell on this issue briefly below, but mostly ignore it for the purpose(s) of this course. Thus, here (as in much of the remaining course material) we assume that we have accomplished this first step and that we want to carry out some calculation with our system, e.g., a minimization. Let's use crambin (PDB entry: 1CRN), and all files should be in your working directory. The actual script is <tt>1crn_1.inp</tt>, and we'll take a look at it shortly.  Before we can dig in, we need to say a few words about CHARMM data structures

#### 2.1 Some comments about CHARMM data structures (files needed by CHARMM)

**RTF (residue topology file):**  The so-called ''residue topology file'' contains the information how the building blocks of larger molecules (e.g., amino acids in the case of proteins) look like. In the context of CHARMM the term ''residue'' is used not only for amino acids, but also, e.g., for nucleic acids or sugars, as well as small molecules, e.g., water.  For each residue, the RTF provides information about the atoms it contains, the atom type, and the atomic charges. Further, it specifies the connectivity of the atoms (which atom is (chemically) bonded to what other atoms). From these data, CHARMM can usually derive information about angles and dihedral angles itself, although one can/could also specify them by hand. CHARMM has RTFs for nucleic acids, lipids, proteins and carbohydrates, and more. E.g., the file <tt>toppar/top_all22_prot.rtf</tt> is part of the CHARMM distribution and contains residue information for amino acids parametrized in accordance with the CHARMM22/27 force field. Without going into details, a short look at the file should give you an idea of the type of information it contains. The <tt>toppar</tt> directory contains similar files for other classes of compounds, e.g., nucleic acids (<tt>top_all27_na.rtf</tt>). (This example uses CHARMM22, a somewhat dated choice of protein force field -- CHARMM36/36m is the modern default -- though for an introductory minimization exercise like this one, it makes no real difference.)

**Parameters:** The force field parameters in (more or less) human readable form; e.g., force  constants of the bond stretching terms, Lennard-Jones radii and  well-depths for all atom types etc. (The notable exception are partial atomic charges, which are listed in the RTF --- can you think of the reason why?!) E.g., the file <tt>toppar/par_all22_prot.prm</tt> is the companion of <tt>toppar/top_all22_prot.rtf</tt> and contains parameters for all compounds described by this RTF file (amino acids). Again, a short glimpse at the file should give you an idea of the info it contains.

**PSF (protein structure file):** The RTF and parameter files (and once processed by CHARMM the corresponding data structures) contain information that applies to, e.g., all proteins by specifying how amino acids look like. Obviously, you want to simulate a specific protein, characterized by its unique primary sequence. The PSF datastructure contains exactly this system specific information, such as the exact list of all bond stretching terms in your protein, all angle bending terms etc. and much more. Before you can do anything useful with CHARMM, this information must have been created internally (we'll see later how this is done). Once created, the PSF data structure can be saved to a file for later reuse; this is what we shall be doing in our crambin example. Note that the PSF does ''not'' contain the force field parameters. As just mentioned, all parameters are read from a parameter file and stored in (an) appropriate data structure(s). The parameters that are needed for the specific system of interest (e.g. crambin) are internally stored in something called the CODES data structure. Normally, you don't have to be concerned with this; however, if you get a CODES error, this normally indicates that your parameter file was incomplete or inconsistent.

**Coordinates:** Finally, you need coordinates for each atom in the system. The most frequent source of coordinates in real work (with proteins) are PDB (protein database) files, but they need some "massaging" before they can be used by CHARMM. Therefore, we use preprepared coordinates for our example.

Normally, one RTF and parameter file suffices for a complete class of systems (e.g., proteins), and these files are provided together with CHARMM. (More formally, together with the form of the potential energy function, the RTF and parameter file constitute what is referred to as "the CHARMM force field".  PSF and coordinates, on the other hand, are specific to the system being studied and  have to be created by the user.


#### 2.2 Back to the crambin minimization example 

Provided one has a PSF and coordinates for the system of interest, many typical applications of CHARMM follow a rather simple pattern:
* (A) Read RTF and parameter file (i.e., load force field information into CHARMM)
* (B) Read PSF and coordinate file (i.e., load information about specific system)
* (C) If necessary, do some more preparatory stuff
* (D) Do "some work"
* (E) Possibly do "on the fly" analysis of what was done in (D)
* (F) Frequently, (D) changed the coordinates; so save the coordinates and whatever else may have changed. 

##### 2.2.1 Reading files

Please do take a look in the CHARMM script <tt>1crn_1.inp</tt> now (open this file with the editor). As you may expect, it contains a title; followed by a lengthy comment paraphrasing what was just said. To read the RTF and parameter information, we shall use a shortcut in the context of this course. The first real command just reads

    stream toppar.str

and this means ''continue reading commands from a file <tt>toppar.str</tt>. When you are done, return and continue reading from this file (<tt>1crn_1.inp</tt>)''. <tt>toppar.str</tt> actually reads a rather full set of RTF/parameter information into CHARMM, after it has concluded we could simulate proteins, DNA, RNA, membranes etc. One can really view this as a black box, and so we ignore this step, essential as it may be. Be warned, however, that processing <tt>toppar.str</tt> generates many, many lines of output, including strange warnings;  these can all be ignored!!

Next, we read the PSF file, which is accomplished by the following command:

    read psf card name 1crn.psf

Obviously, you have to specify the filename (<tt>name</tt> \<filename\>). The keyword <tt>read</tt>
is obvious, the argument <tt>psf</tt> indicates what file type is to be read, and the option
<tt>card</tt> is CHARMM syntax for denoting that CHARMM's own format is to be used. 

Aside: In many CHARMM scripts you'll find the older way of reading (writing) files, which consists
of the following three statements (effectively equilent to the one line above):

    open unit 1 form read name 1crn.psf
    read psf card unit 1
    clos unit 1

Since there are instances where the long form is advantageous we may just as well discuss it here:
Here I/O (Input/Output) follows the model of most programming languages; since CHARMM is programmed in FORTRAN, there are similarities to the FORTRAN I/O model. First, a file has to be ''opened'' ("connected"  to a "unit"); the remaining arguments to the open command specify the mode of operation. Note: units 5 and 6 are reserved; on most systems you can have unit numbers ranging from 1-4 to 7-99. Obviously, you have to specify the file name (<tt>name \<name\></tt>). The keyword <tt>read</tt> indicates that the file is opened for read access (meaning that it is an error if the file does not exist!). Further, we specify that the file is "formatted" (keyword <tt>form</tt> (alternatively one could use  <tt>card</tt>)), i.e., it is an ASCII text file as opposed to a binary file (in which case the keyword would be <tt>file</tt>; <tt>file</tt> is actually the default, so <tt>form</tt> must be specified!). 

The next command (<tt>read rtf card unit 1</tt>) does the actual reading. The first keyword after <tt>read</tt> specifies what type of file is going to be read; not surprisingly, the keyword is <tt>psf</tt> for PSFs, <tt>coor</tt> for coordinate files etc. (see 1crn_1.inp). The second keyword after the <tt>read</tt> command has to do with the fact that many types of files can be read by CHARMM in more than one format (we'll see examples later!). In all <tt>read</tt> statements in 1crn_1.inp we specify the respective native CHARMM format by the <tt>card</tt> keyword.  Finally, we have to tell CHARMM from which <tt>unit</tt> we want to read; typically (as is the case here) this is the unit that was opened just before the <tt>read</tt> command (<tt> read ... unit 1</tt>). 

Finally, files opened for reading, should be explicitly closed when one is done. (This is not necessary, but good practice).


##### 2.2.2 Do some work

It's time that you should run the script

    charmm -i 1crn_1.inp > 1crn_1.out

and look also at the output. As mentioned, <tt>toppar.str</tt> produces lots of output; ignore it and search for the reading of the PSF file. Shortly afterwards the minimization starts, we'll look into what's going on here in some detail further down. Clearly, some work is being done.

##### Writing files

At the end of the minimization, our system does not only have a much lower energy, but its coordinates have changed as well. Most likely, one wants to save these coordinates for later use, and so we have to learn how to write files.  Writing files in CHARMM is very similar to reading files.
Using the short form, you specify

    writ coor card name 1crn_somemini.crd

If you look at <tt>1crn_1.inp</tt>, you'll see that we actually write the coordinates twice, once in CHARMM format (keyword <tt>card</tt>, but also in PDB format (keyword <tt>pdb</tt>). The native CHARMM format is intended for reuse by CHARMM, whereas the PDB file can be used for data exchange with other programs, in particular (but not limited to) graphics programs. The novel part (compared to reading) is that the <tt>writ</tt> command is followed by a title

    writ coor pdb name 1crn_somemini.pdb
    * CRAMBIN, from pdb entry 1crn,
    * with CHARMM27 parameter set
    * after 100 steps SD + 100 steps ABNR minimization
    *

This provides the possibility to pass some information into the respective output file (take a look at <tt>1crn_somemini.crd</tt>, the title is there, augmented by some "accounting information"). The title can be ommitted, but providing an informative(!) title is good practice.

Alternatively, you can also use the long form as discussed for reading, i.e., an <tt>open</tt> statement connecting a file to a unit, followed by writing to that unit (the title gets specified as shown above). After most type of writes, files are closed automatically; so there is usually no need for an explicit <tt>clos</tt> command.


##### 2.2.3 Closing comments

Please make sure that you understand the basic structure of this script, which could serve / is a template for most real-life charmm scripts. Note that the real work is done by the two <tt>mini</tt> lines. This is not too untypical; setting up the system, setting up options takes often many more comments than initiating the real computational work.

Better late than never: CHARMM commands are case-insensitive, i.e., it does not matter whether you write <tt>WRITE</tt>, <tt>write</tt>, or <tt>WriTE</tt>. Further, in most instances only the first four (4) letters of a command are relevant, i.e.,

    ENER

and 

    energy

are completely equivalent. There are commands that accept (too) many options, and in some cases the first six (6) letters of the options are relevant. The most obvious I can think of are the options to the DYNAmics command (the command for running molecular dynamics), but since we are not going to use it in this course, this detail is of little interest. 

So, commands are case-insensitive, but the operating system, in particular the file system are case sensitive, i.e., under Unix/Linux <tt>input.txt</tt>, <tt>INPUT.TXT</tt> and <tt>input.TXT</tt> denote different files. My advice when working with CHARMM is to use only lower case letters in directory and file names. For file names (including the path), CHARMM tries automatically the all uppercase and the all lowercase option. If you really  want (or have to) use mixed case names, then file names have to be surrounded by "". Example:

    read .. name data.stuff

would read from a file named either <tt>data.stuff</tt> or <tt>DATA.STUFF</tt>. If, however, your filename is
<tt>DATA.stuff</tt>, then the read statement should read

    read .. name "DATA.stuff"

The same applies to the <tt>open</tt> command used in the long form of reading/writing.


### 3. Setting up a pdb file for calculations with CHARMM

When visualizing a molecule, a pdb file is all you need. Clearly, you need more information to compute energies etc.; also, when doing so the information must not be ambiguous. pdb file may lack coordinates of some residues, they usually lack all hydrogen positions (why). You need to decide protonation states, particularly of histidines, you need to decide whether to keep crystal waters etc. Specifically for CHARMM, you face the added complication that the pdb reader is dumb, so you have to massage the pdb file prior to 'feeding' it to charmm. This used to be a major barrier for setting up simulations; by now there are some tools to help you. In principle,
[vmd](http://www.ks.uiuc.edu/Research/vmd/plugins/autopsf/) or the newer [QwikMD](http://www.ks.uiuc.edu/Research/qwikmd/) plugin can carry out /assist you in the necessary steps. 
For various reasons we prefer the [charmm-gui server](http://www.charmm-gui.org). 

We will return to this aspect (setting up a system for MD) at the end of this course. Meanwhile, if you are curious go to http://www.charmm-gui.org/?doc=input/pdbreader and enter 1CRN in the "Download PDB File" box, and click through the steps. (**Remember: you need an account!** Use 'PDB Reader' from the input generator section!) You may download the final tar.gz package. The PSF and CRD files generated by the site are actually the files I used in the initial crambin example (though they were generated some years ago, so they will not be identical!); similarly, the <tt>toppar.str</tt> setup was taken from charmm-gui. Feel free to look at the script <tt>step1_pdbreader.inp</tt>; we can discuss its contents in class; understanding it is ''not'' required for this course!

### 4. Energy calculation and minimization

Here we look a little bit behind the scenes of energy calculation and minimization. We start by discussing the calculation of energy and forces, followed by some comments on the various minimizers.

#### 4.1 Energy calculation

A few remarks have to suffice here; please let me know if these concepts have not been dealt with in the lecture!

To calculate the energy of a system, CHARMM needs (cartesian) coordinates for each atom and parameters for each energy term. The molecular mechanics force field used by CHARMM and similar computer programs is a (very) simplified, parametrized representation of reality. Interactions between chemically bonded nearest neighbors are handled by special terms, the bonded energy terms: bond stretching, angle bending, dihedral and improper dihedral energy terms. In the new all-atom CHARMM force-field, there is also a harmonic term acting between 1-3 neighbors, the so-called Urey-Bradley term. It's not present in the polar hydrogen force-field. Interactions other than those between chemically bonded nearest neighbors are represented by the two nonbonded energy terms: Lennard Jones and electrostatic. Early versions of the force-field contained a special term for hydrogen bonds; the code is still in CHARMM, but it is, in general, not used anymore. To compute the interaction of N atoms with each other, one needs in principle <math>N^2</math> steps. Bonded interactions only involve next neighbors, so they are very straightforward and cheap in terms of computer time. For the nonbonded energy (Lennard-Jones, electrostatic) it is customary to introduce a cut-off beyond which interactions are ignored. This is a good approximation for the van der Waals interactions, which decay rapidly for large distances,
but a bad one for electrostatic interactions which go to zero as <math>1/r</math>. To mitigate the side effects of truncating
interactions at the cut-off distance, so-called ''shifting'' or ''switching'' functions are used to turn off interactions more smoothly. More advanced methods, such as the Ewald summation method, a clean and efficient solution for this problem for periodic
systems used in the study of liquids and solute-solvent systems is beyond the scope of this course. From a practical point of view, ''implicit solvent'' models can be viewed as a combination of additional energy terms with highly specialized "shifting"/"switching" functions.

To avoid computing interactions between chemically bonded atoms (1-2, 1-3 and, possibly, 1-4 neighbors) twice (nonbonded interactions in addition to the bonded terms), these pairs are excluded from the nonbonded energy  calculation. It was already mentioned, that frequently a cut-off criterion is used to limit the number of interactions. Energy minimizations and especially molecular dynamics simulations require hundreds and thousands of energy calculations. To avoid checking for excluded pairs (1-2, 1-3 and, possibly, 1-4 neighbors) and, more
importantly, the cut-off criterion during every single energy calculation, lists are generated that contain only those pairs for which the nonbonded interactions  should be computed. This is an unnecessary overhead for a single calculation, but saves a lot of time during repeated computations. CHARMM (and, I guess,
most other molecular mechanics packages) always has to generate these nonbonded lists before an energy computation can take place, and these lists need to be updated during repeated energy calculations.

In the output of the "first example" discussed above, which performs a minimization, following the ENERgy and MINImize commands you will find lines giving you information about the numerous options that were used for the energy calculation; these look something like:

    NONBOND OPTION FLAGS: 
      ELEC     VDW      ATOMs    CDIElec  SHIFt    VATOm    VSWItch 
      BYGRoup  NOEXtnd  NOEWald 
    CUTNB  = 14.000 CTEXNB =999.000 CTONNB = 10.000 CTOFNB = 12.000
    CGONNB =  0.000 CGOFNB = 10.000
    WMIN   =  1.500 WRNMXD =  0.500 E14FAC =  1.000 EPS    =  1.000
    NBXMOD =      5

Some explanations:ELEC and VDW just mean that both electrostatic and Lennard-Jones interactions
are computed. The WMIN distance is used to print out interaction pairs that are closer than 1.5 A. At shorter distances, very high Lennard-Jones repulsion would result, and it's useful to be warned about such pairs. The NBXMOD=5 entry is the CHARMM way of saying that 1-2 and 1-3 neighbors should be excluded, and 1-4 pairs are treated specially. In the parameter file, there is the option to have a separate set of Lennard-Jones parameters for this case; the electrostatic energy gets computed normally, but is scaled by E14FAC (in the <tt>1crn_1.inp</tt> example E14FAC is 1.0). For the old polar hydrogen set E14FAC 0.4 was the proper value to use, for the all-atom force field used here E14FAC=1.0; fortunately, this is set automatically; never overrule this value unless you know very well what you are doing. BYGR specifies the algorithm used to build the nonbonded pairlist; this is fine for systems in the gas phase or when using implicit solvent. For large solvated systems under periodic boundary conditions, it should be replaced by the BYCB option! NOEW means that the Ewald summation method is not used (it is not applicable to a system without solvent). CUTNB is the cut-off radius used to construct the pairlist, whereas CTOFNB is the cut-off radius used in the energy calculation. This allows for a pair to be included in the nonbonded list; nevertheless, the interaction may be excluded from the energy calculation itself if the distance between the two atoms falls between 12 and 14 A (in the above snippet). The region between the two values serves as a buffer. Remember that the nonbonded list is not re-built for each energy evaluation. During minimization, molecular dynamics or Monte Carlo simulations the atom positions and hence the distances between pairs change. The extra pairs in the nonbonded list help to avoid errors in the energy calculation due to movement of atoms. Obviously, the nonbonded list needs to be updated at reasonable intervals, by now the default in CHARMM is to do this automatically based on a heuristic scheme.

The remaining options govern additional, important details of the nonbonded energy calculation. First, the cut-off criterion can be applied on an atom by atom basis (ATOM, VATOm) or on a group by group basis (GROUp, VGROup). The option with V in front pertains to the van der Waals energy computation, that
without to the electrostatic energy computation. The atom by atom case is pretty self-explanatory. Alternatively, cut-offs can be applied on a group by group basis. In the RTF files, there are special indicators placing neighboring atoms into ''groups''. This information is used if the GROU and VGRO option is set for the energy calculation. In this case, CHARMM first computes the center of geometry for every group. If the distance between the centers of geometry for a pair of groups are separated by more than the cut-off distance, then no interactions between any atoms of the two groups are computed. By contrast, in the atom by atom case, some pairs of atoms might be inside and others might be outside the cut-off distance. Note, you either have to use GROU/VGRO or ATOM/VATO, you cannot mix! Thus, in general, the two options give different results. In most practical work with CHARMM ATOM and VATO are used (these are also the default options).

The straightforward application of a cut-off criterion introduces a discontinuity that can adversely affect the stability of a molecular dynamics calculation or the result of a minimization. Consider a pair of atoms that are separated by approximately the cut-off distance. At one timestep they interact and contribute to the energy and forces in the system. Then they move relative to each other by a tiny amount and suddenly they are more than the cut-off distance apart.  Although very little has changed, they now do not contribute to the energy and
forces at all. To avoid these abrupt changes, the Lennard-Jones and electrostatic energy functions in CHARMM are tapered by switching (SWITCH, VSWITCH, VFSWITCH) and/or shifting functions (SHIFT, VSHIFT). A switching function requires a second cut-off to be specified; this is the CTONNB value printed by CHARMM. For distances less than this inner threshold, the nonbonded energy is computed without modification. Between the two distances, the energy (and forces) is (are) scaled by a function that goes smoothly to zero as the separation approaches
the outer cut-off; interactions beyond the outer cut-off are ignored, as always. A shifting function modifies the energy function over the full range of distance (i.e., CTONNB is not needed) so that instead of
approaching zero as the distance r goes to infinity, it is zero at the cut-off distance (CTOFNB). By comparison, shifting is smoother than switching; however, the energy and forces are modified (subtly) for
all distances whereas a switching function only affects the range between inner and outer cut-off. For the Lennard-Jones interaction (force) switching is now the preferred method (V(F)SWITCH).  None of the methods is truly appropriate for electrostatic interactions; it seems, however, as if
shifting gives the better results in this case. Again, for solvated systems the Ewald summation method avoids the need for such kludges. If one has to use a cut-off for electrostatic interactions (e.g., for
gas phase calculations), one should set a reasonably large cutoff. The default of the new all-atom force-field used in the <tt>1crn_1.inp</tt> example is 12A, which is more reasonable. 

Two related options remain to be explained: CDIEl and EPS. The solvent (water) is a very important determinant when studying the properties of biomolecules.  However, calculations of solute-solvent systems are
very costly in terms of computer time, so even nowadays biomolecules are sometimes studied in the gas phase. Water has a high dielectric constant (DC), which screens electrostatic interactions. In a
solute-solvent system, the DC is a property of the system (which can, in fact, be computed from a MD simulation). To mimic the presence of water, CHARMM allows the user to change the value of the DC by the EPS
keyword (default value is 1), i.e., the electrostatic energy is computed according to $q1\times q2/(\mathrm{EPS}\times r)$. The CDIEl / RDIEl option is another attempt to mimic water.  CDIE stands for
constant (uniform) dielectric, RDIE means a distance-dependent dielectric. Effectively, when you use RDIE, you compute a modified electrostatic energy according to $q1\times q2 / (\mathrm{EPS}\times
r^2)$. Needless to say, none of the two approaches is very realistic.  During the last 15 years, these simplistic approaches have been complemented by a multitude of implicit solvent models; e.g., JCC (2003), 24, 1348.

#### 4.2 Minimization 

CHARMM offers the choice of several minimizers; the theoretical basis of at least some of them was presented in the lecture. By far the simplest method is the steepest descent algorithm (SD), where one simply follows the gradient of the potential, multiplied by a factor (the step size (STEP)). It is the best method if you are very far away from a minimum. The closer one gets to a minimum, the worse the performance, in fact, the algorithm may get trapped and oscillate back and forth endlessly. The conjugate gradient method (CONJ), the Powell
algorithm (POWE) and the adopted basis set Newton Raphson (ABNR) minimizer all attempt to overcome this efficiency by taking into account the previous steps during the minimization. Powell and ABNR are the recommended methods for most applications.  Finally, the Newton-Raphson method (NRAP) is the most elaborate of all minimizers.
It requires the second derivatives of the energy as well, which increases the amount of computer memory and the times of computations tremendously. Thus, the NRAP method cannot be applied to large systems (crambin is OK on a modern machine, but extremely slow) Often, a combination of methods is used. First, a few steps of SD, followed by
a larger number of one of the more intelligent algorithms (CONJ, POWE, ABNR). Finally, if the system size allows it, one may conclude with a few steps of NRAP. 

A hint to conclude the general discussion. No computer does calculations internally in exactly the same manner. Thus, if one runs 10,000 steps of molecular dynamics on different generations of Intel processors, or on different types of processors, one will in general obtain slightly different results (usually, the numbers start to diverge noticeably after 500--1000 steps). The same may happen on identical machines if different versions of the compiler were used to compile CHARMM. Minimization is particularly prone to this effect, so depending on the machine you may get slightly different results during minimization, either in the final value or in the number of steps that are required to obtain converged results. (During minimization, differences can occur already after a single step!) Let me stress that this is not a bug in CHARMM, but that it is due to the structure of minimization algorithms and arithmetic with a finite precision.

<font color="red">**4.3 EXERCISE TIME:**</font> After all this talk, let's try out a few things. The script <tt>minmiz1.inp</tt> minimizes N-acetyl-L-tryptophanemethylamide, i.e., a tryptophane with the amino as well as the carboxylic acid end blocked.  Most of the script is rather uninteresting (the details how the peptide coordinates are generated are not important and hidden in the minmiz_setup.str streamfile).  As written, the script <tt>minmiz1.inp</tt> performs 100 steps of SD minimization. Commented out are some suggestions for the use of the other minimizers. Try
out all of them. For the SD algorithm, can you understand the adaption of the step-size? Compare the efficiency (what is the final energy) and CPU-time used (see the last two lines of the CHARMM output) of the various methods. (Some timings and memory statistics are given at the end of a CHARMM output file).

On the machine I tested this program, only the NRAP method converged to the desired tolerance in less than 100 steps. You may want to increase the number of steps for the other methods and see whether you can reach the NRAP result. The script also illustrates that CHARMM can be used to compute RMSD differences.

<font color="red">Nothing to hand in for this unit! After working your way to this point, you should be

 + able to run a CHARMM job
 + identify some basic elements of a CHARMM input
 + be able to make minor modifications to example inputs given in later units
</font>